# Подготовка данных для статьи - исследования "Секреты Темнолесья"

Цель проекта - привлечь новую аудиторию в игру "Секреты Темнолесья". Для этого команда напишет развлекательно-познавательную статью на сайте известного IT ресурса. Статья - исследование об истрии игр в 21 веке. Задачи проекта - изучить развитие игровой индустрии с 2000 по 2013 год и подготовить качественные данные для написания статьи.

Шаги:
Изучить собранные данные
Проверить корректность данных и сделать полную предобработку
Отфильтровать для получения нужного среза данных
Выделить ТОП-7 платформ лидеров по количеству вышедших игр за 2000 - 2013 годы

__Описание данных:__

Данные содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр: 
- Name — название игры
- Platform — название платформы
- Year of Release — год выпуска игры
- Genre — жанр игры
- NA sales — продажи в Северной Америке (в миллионах проданных копий)
- EU sales — продажи в Европе (в миллионах проданных копий)
- JP sales — продажи в Японии (в миллионах проданных копий)
- Other sales — продажи в других странах (в миллионах проданных копий)
- Critic Score — оценка критиков (от 0 до 100)
- User Score — оценка пользователей (от 0 до 10)
- Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

In [1]:
import pandas as pd
df = pd.read_csv('/Users/ekaterinaproshenkova/Downloads/new_games.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


__Комментарий:__ 
1. В датасете 11 столбцов и 16956 строк
2. Названия столбцов соответствуют описанию
3. Пропуски:
Name - 2, 
Year of Release - 275, 
Genre - 2, 
Critic Score - 8342(!), 
User Score - 6804(!), 
Rating - 6871(!)
4. Типы данных: нужно преобразовать EU sales, JP sales, User Score из 'object' в 'float64'. Year of Realise привести к 'int64'. Rating оставляем в 'object' так как там буквенные обозначения.

In [2]:
print(df.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')


In [3]:
# Привожу названия стоблцов к единому стилю snake_case

df.columns = df.columns.str.replace(' ', '_').str.lower()
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')


### Преобразование типов данных

In [4]:
# Приведу типы данных в столбцах eu_sales (продажи в Европе) и jp_sales (продажи в Японии) к float64

df['eu_sales'] = pd.to_numeric(df['eu_sales'].str.replace(',', '.'), errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'].str.replace(',', '.'), errors='coerce')

# Приведу типы данных в столбце user_score к float64

df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')

print(df.info)

<bound method DataFrame.info of                                 name platform  year_of_release         genre  \
0                         Wii Sports      Wii           2006.0        Sports   
1                  Super Mario Bros.      NES           1985.0      Platform   
2                     Mario Kart Wii      Wii           2008.0        Racing   
3                  Wii Sports Resort      Wii           2009.0        Sports   
4           Pokemon Red/Pokemon Blue       GB           1996.0  Role-Playing   
...                              ...      ...              ...           ...   
16951  Samurai Warriors: Sanada Maru      PS3           2016.0        Action   
16952               LMA Manager 2007     X360           2006.0        Sports   
16953        Haitaka no Psychedelica      PSV           2016.0     Adventure   
16954               Spirits & Spells      GBA           2003.0      Platform   
16955            Winning Post 8 2016      PSV           2016.0    Simulation   

       

__Комментарий:__
Столбец Year of Release -тип float64. В столбце 275 пропусков, пропуски не могут быть int64, поэтому автоматически приводятся к float64. Преобразую позже, так как это не критично и мы не можем заполнить год выхода игры средним значением или медианой.

### Работа с пропусками

In [5]:
# Считаю количество пропусков в каждом столбце в абсолютных и относительных значениях

missing_abs = df.isnull().sum()
missing_rel = (df.isnull().sum() / len(df)) * 100
missing_info = pd.DataFrame({
    'Абсолютные пропуски': missing_abs,
    'Относительные пропуски (%)': missing_rel.round(2)
})
print(missing_info)

                 Абсолютные пропуски  Относительные пропуски (%)
name                               2                        0.01
platform                           0                        0.00
year_of_release                  275                        1.62
genre                              2                        0.01
na_sales                           0                        0.00
eu_sales                           6                        0.04
jp_sales                           4                        0.02
other_sales                        0                        0.00
critic_score                    8714                       51.39
user_score                      9268                       54.66
rating                          6871                       40.52


__Комментарий:__ Пропуски в столбцах name (2 пропуска), genre (2 пропуска), eu_sales (6 пропусков), jp_sales (4 пропуска) можно удалить. Критически много пропусков в столбцах Сritical_score - 8714 пропуска, user_score - 9268, rating - 6871. Гипотеза большого количества пропусков в том, что для слишком старых игр или для игр из других регионов, не переведенных на английский язык, не существует достаточного количества оценок критиков и пользователей по естественным причинам. Такой большой срез данных нельзя удалять, заполним его специальным значением -1.

In [6]:
# Удаляю пропуски в столбцах name, genre, eu_sales, jp_sales

df = df.dropna(subset=['name', 'genre', 'eu_sales', 'jp_sales'])
df.info()

<class 'pandas.DataFrame'>
Index: 16944 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16944 non-null  str    
 1   platform         16944 non-null  str    
 2   year_of_release  16669 non-null  float64
 3   genre            16944 non-null  str    
 4   na_sales         16944 non-null  float64
 5   eu_sales         16944 non-null  float64
 6   jp_sales         16944 non-null  float64
 7   other_sales      16944 non-null  float64
 8   critic_score     8234 non-null   float64
 9   user_score       7680 non-null   float64
 10  rating           10076 non-null  str    
dtypes: float64(7), str(4)
memory usage: 1.6 MB


In [7]:
# Заполняю пропуски в столбцах critic_score, user_score и year_of_release значением -1

df['critic_score'] = df['critic_score'].fillna(-1)
df['user_score'] = df['user_score'].fillna(-1)
df['year_of_release'] = df['year_of_release'].fillna(-1)
df.info()

<class 'pandas.DataFrame'>
Index: 16944 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16944 non-null  str    
 1   platform         16944 non-null  str    
 2   year_of_release  16944 non-null  float64
 3   genre            16944 non-null  str    
 4   na_sales         16944 non-null  float64
 5   eu_sales         16944 non-null  float64
 6   jp_sales         16944 non-null  float64
 7   other_sales      16944 non-null  float64
 8   critic_score     16944 non-null  float64
 9   user_score       16944 non-null  float64
 10  rating           10076 non-null  str    
dtypes: float64(7), str(4)
memory usage: 1.6 MB


In [8]:
# Приведу year_of_release к int64

df['year_of_release'] = df['year_of_release'].astype('int64')
df.info

<bound method DataFrame.info of                                 name platform  year_of_release         genre  \
0                         Wii Sports      Wii             2006        Sports   
1                  Super Mario Bros.      NES             1985      Platform   
2                     Mario Kart Wii      Wii             2008        Racing   
3                  Wii Sports Resort      Wii             2009        Sports   
4           Pokemon Red/Pokemon Blue       GB             1996  Role-Playing   
...                              ...      ...              ...           ...   
16951  Samurai Warriors: Sanada Maru      PS3             2016        Action   
16952               LMA Manager 2007     X360             2006        Sports   
16953        Haitaka no Psychedelica      PSV             2016     Adventure   
16954               Spirits & Spells      GBA             2003      Platform   
16955            Winning Post 8 2016      PSV             2016    Simulation   

       

### Работа с явными и неявными дубликатами

In [9]:
# Проверю уникальные значения в категориальных данных

for column in ['platform','year_of_release','genre','rating']:
    print(f'Уникальные значения в столбце {column}:')
    print(df[column].sort_values().unique())
    print()

Уникальные значения в столбце platform:
<StringArray>
['2600',  '3DO',  '3DS',   'DC',   'DS',   'GB',  'GBA',   'GC',  'GEN',
   'GG',  'N64',  'NES',   'NG',   'PC', 'PCFX',   'PS',  'PS2',  'PS3',
  'PS4',  'PSP',  'PSV',  'SAT',  'SCD', 'SNES', 'TG16',   'WS',  'Wii',
 'WiiU', 'X360',   'XB', 'XOne']
Length: 31, dtype: str

Уникальные значения в столбце year_of_release:
[  -1 1980 1981 1982 1983 1984 1985 1986 1987 1988 1989 1990 1991 1992
 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005 2006
 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016]

Уникальные значения в столбце genre:
<StringArray>
[      'ACTION',    'ADVENTURE',       'Action',    'Adventure',
     'FIGHTING',     'Fighting',         'MISC',         'Misc',
     'PLATFORM',       'PUZZLE',     'Platform',       'Puzzle',
       'RACING', 'ROLE-PLAYING',       'Racing', 'Role-Playing',
      'SHOOTER',   'SIMULATION',       'SPORTS',     'STRATEGY',
      'Shooter',   'Simulation',       'Sports',    

In [10]:
# Приведу Genre к нижнему регистру и удалю лишние пробелы
df['genre'] = df['genre'].str.lower().str.strip()

# Приведу Rating к верхнему регистру и удалю лишние пробелы
df['rating'] = df['rating'].str.upper().str.strip()

In [11]:
# Проверяю на явные дубликаты 
full_duplicates = df.duplicated()
print(f"Полных дубликатов строк: {full_duplicates.sum()}")

if full_duplicates.sum() > 0:
    print("\nПримеры полных дубликатов:")
    print(df[full_duplicates].head())

Полных дубликатов строк: 241

Примеры полных дубликатов:
                                 name platform  year_of_release    genre  \
268             Batman: Arkham Asylum      PS3             2009   action   
368  James Bond 007: Agent Under Fire      PS2             2001  shooter   
717             God of War: Ascension      PS3             2013   action   
823                 Wipeout: The Game      Wii             2009     misc   
848   Rayman Raving Rabbids: TV Party      Wii             2008     misc   

     na_sales  eu_sales  jp_sales  other_sales  critic_score  user_score  \
268      2.24      1.31      0.07         0.61          91.0         8.9   
368      1.90      1.13      0.10         0.41          72.0         7.9   
717      1.23      0.63      0.04         0.35          80.0         7.5   
823      1.94      0.00      0.00         0.12          -1.0        -1.0   
848      0.72      1.08      0.00         0.23          73.0         7.7   

    rating  
268      T  
368

In [12]:
# Проверяю количество строк до удаления
print(f"Количество строк до удаления дубликатов: {len(df)}")

# Проверяю наличие полных дубликатов
full_duplicates_count = df.duplicated().sum()
print(f"Найдено полных дубликатов: {full_duplicates_count}")

# Удаляю полные дубликаты
if full_duplicates_count > 0:
    df = df.drop_duplicates()
    print(f"Удалено {full_duplicates_count} полных дубликатов")
else:
    print("Полные дубликаты не найдены, удаление не требуется")

# Проверяю количество строк после удаления
print(f"Количество строк после удаления дубликатов: {len(df)}")

Количество строк до удаления дубликатов: 16944
Найдено полных дубликатов: 241
Удалено 241 полных дубликатов
Количество строк после удаления дубликатов: 16703


In [13]:
# Проверяю результат удаления данных по отношению к исходному файлу

initial_count = len(pd.read_csv('/Users/ekaterinaproshenkova/Downloads/new_games.csv'))
print(f"Исходное количество строк: {initial_count}")

final_count = len(df)

removed_count = initial_count - final_count
removed_percentage = (removed_count / initial_count) * 100

print(f"Абсолютное количество удаленных строк: {removed_count}")
print(f"Относительное количество удаленных строк: {removed_percentage:.2f}%")
print(f"Осталось строк: {final_count}")

Исходное количество строк: 16956
Абсолютное количество удаленных строк: 253
Относительное количество удаленных строк: 1.49%
Осталось строк: 16703


__Комментарий:__
Удалила незначительные пропуски и обработала столбцы с критическим количеством пропусков. Названия платформ и жанры стилистически обработала (привела к единому регистру). Провела обработку явных и неявных дубликатов (полные дубликаты удалила). Преобразовала столбец year_of_release к корректному типу данных int64.

### Фильтрация данных за период с 2000 по 2013 годы включительно

In [14]:
df_actual = df[df['year_of_release'].between(2000,2013)].copy()

### Категоризация данных по трём категориям (оценка пользователей)

In [15]:
# Выделяем три категории: высокая оценка, средняя оценка, низкая оценка

def categorize_user_score(score):

    if score >= 8:
        return 'высокая оценка'
    elif score >= 3:
        return 'средняя оценка'
    else:
        return 'низкая оценка'

df_actual['user_score_category'] = df_actual['user_score'].apply(categorize_user_score)

print("\nРаспределение по категориям:")
print(df_actual['user_score_category'].value_counts())


Распределение по категориям:
user_score_category
низкая оценка     6412
средняя оценка    4078
высокая оценка    2282
Name: count, dtype: int64


### Категоризация данных по трём категориям (оценка критиков)

In [16]:
def categorize_critic_score(score):
    if score >= 80: 
        return 'высокая оценка'
    elif score >= 30: 
        return 'средняя оценка'
    elif score >= 0:
        return 'низкая оценка'

df_actual['critic_score_category'] = df_actual['critic_score'].apply(categorize_critic_score)

print("\nКоличество игр в каждой категории:")
print(df_actual['critic_score_category'].value_counts())

print("\nСредняя оценка по категориям:")
print(df_actual.groupby('critic_score_category')['critic_score'].mean().round(1).sort_values(ascending=False))


Количество игр в каждой категории:
critic_score_category
средняя оценка    5421
высокая оценка    1686
низкая оценка       55
Name: count, dtype: int64

Средняя оценка по категориям:
critic_score_category
высокая оценка    85.1
средняя оценка    63.8
низкая оценка     24.9
Name: critic_score, dtype: float64


### Вывод единого результата и подсчёт количества игр в каждой категории за период с 2000 по 2013 годы

In [17]:
 # Группировка по категориям оценок пользователей
user_score_groups = df_actual.groupby('user_score_category').size()
print("Количество игр по категориям пользовательских оценок:")
for category, count in user_score_groups.items():
    percentage = (count / len(df_actual)) * 100
    print(f"- {category}: {count} игр ({percentage:.1f}%)")

# Группировка по категориям оценок критиков
critic_score_groups = df_actual.groupby('critic_score_category').size()
print("Количество игр по категориям оценок критиков:")
for category, count in critic_score_groups.items():
    percentage = (count / len(df_actual)) * 100
    print(f"- {category}: {count} игр ({percentage:.1f}%)")

total_games = len(df_actual)
print(f"Общее количество игр в периоде 2000-2013: {total_games}")

Количество игр по категориям пользовательских оценок:
- высокая оценка: 2282 игр (17.9%)
- низкая оценка: 6412 игр (50.2%)
- средняя оценка: 4078 игр (31.9%)
Количество игр по категориям оценок критиков:
- высокая оценка: 1686 игр (13.2%)
- низкая оценка: 55 игр (0.4%)
- средняя оценка: 5421 игр (42.4%)
Общее количество игр в периоде 2000-2013: 12772


### ТОП-7 платформ по количеству игр за период с 2000 по 2013 годы

In [18]:
top_platforms = df_actual['platform'].value_counts().head(7)
print("ТОП-7 ПЛАТФОРМ ПО КОЛИЧЕСТВУ ИГР (2000-2013)")
for i, (platform, count) in enumerate(top_platforms.items(), 1):
    percentage = (count / len(df_actual)) * 100
    print(f"{i}. {platform}: {count} игр ({percentage:.1f}%)")

ТОП-7 ПЛАТФОРМ ПО КОЛИЧЕСТВУ ИГР (2000-2013)
1. PS2: 2126 игр (16.6%)
2. DS: 2117 игр (16.6%)
3. Wii: 1275 игр (10.0%)
4. PSP: 1179 игр (9.2%)
5. X360: 1118 игр (8.8%)
6. PS3: 1087 игр (8.5%)
7. GBA: 810 игр (6.3%)


## Результат предобработки данных

1. Названия столбов приведены к единому стилю snake_case
2. В столбцах eu_sales,  jp_sales, user_score данные приведены к типу numeric
3.  Найдены и обработаны пропуски. Пропуски удалены в столбах name, genre, eu_sales, jp_sales (всего удалено 14 пропусков). Пропуски в столицах critic_score, user_score, year_of_release заменены значением -1.
4. Обработаны явные и неявные дубликаты. Всего удалено 241 строк полных дубликатов.
5. Данные отфильтрованы за нужный период с 2000 по 2013 годы и выделены в отдельный датафрейм df_actual
6. Данные распределены по трём категориям (высокая оценка, средняя оценка, низкая оценка) по оценкам пользователей (user_score) и оценкам критиков (critic_score)
7. Данные сгруппированы, подсчитано общее количество игр за период с 2000 по 2013 годы.
8. Определены ТОП-7 платформ по количеству игр.